In [1]:
# ---------------------------------------------------------------------
# Developing: Simple notebook setup
# ---------------------------------------------------------------------
import pandas as pd
import numpy as np
from icare_risk.scores import evaluate_score

# 1. Quickly mock the exact columns you want to test
df_test = pd.DataFrame({
    'patient_id': [1, 2, 3, 4],
    'AGE_AT_ADMISSION': [45, 72, 80, 55],
    'local_stress_flag': [0, 1, 1, 0]
})

# 2. Write your new feature/score directly in the cell
def calculate_experimental_score(df,
                                 age_col='AGE_AT_ADMISSION',
                                 stress_col='local_stress_flag'):
    rules = [
        {'desc': 'Elderly', 'col': age_col, 'condition': df[age_col] > 65, 'points': 2},
        {'desc': 'Stressed', 'col': stress_col, 'condition': df[stress_col] == 1, 'points': 3}
    ]
    # Use verbose=True to instantly see the audit logs!
    return evaluate_score(df, rules, "EXPERIMENTAL", verbose=True)

# 3. Test it instantly
df_test['new_score'] = calculate_experimental_score(df_test)
display(df_test)

,patient_id,AGE_AT_ADMISSION,local_stress_flag,new_score
0,1,45,0,0
1,2,72,1,5
2,3,80,1,5
3,4,55,0,0


In [6]:
# ----------------------------------------------------
# Developing: Complex notebook setup
# ----------------------------------------------------

import os
import pandas as pd
from pathlib import Path

# -----------------------
# Temporal fix
# -----------------------
# Might be better to enable exact dir in get_lated_data_dir(exact_dir=....).
# If running inside the 'notebooks' folder, step back up to the project root
if Path.cwd().name == 'notebooks':
    os.chdir('..')
    print(f"📂 Changed working directory to project root: {Path.cwd()}")
else:
    print(f"📂 Working directory: {Path.cwd()}")

# -----------------------
# Step 1
# -----------------------

# Import your core tools
from icare_risk.utils import load_yaml_config, get_latest_data_dir
from icare_risk.features import FeaturePipeline
from icare_risk.scripts.b_build_features_icare import prepare_icare_ts, LazyContextDict

# 1. Load Configs
data_config = load_yaml_config("data_config.yaml")
# Load your local override containing the feature you want to test!
feat_config = load_yaml_config("feature_config.yaml", "../config/local_feature_config.yaml")

# 2. Get Real Data Directory
latest_dir = get_latest_data_dir(loaded_config=data_config)
print(f"Using data from: {latest_dir.name}")

# 3. Load Base Data
df_episodes = pd.read_csv(latest_dir / 'icare_episodes_anon.csv').rename(columns={'SUBJECT': 'patient_id'})
df_vitals = pd.read_csv(latest_dir / 'icare_vital_signs_anon.csv', parse_dates=['OBSERVATION_PERFORMED_DT'])
df_labs = pd.read_csv(latest_dir / 'icare_pathology_blood_anon.csv', parse_dates=['SAMPLE_COLLECTED_DT'])

df_ts = prepare_icare_ts(df_vitals, df_labs, data_config)

# 4. Initialize the Lazy Contexts (The Magic Part)
contexts = LazyContextDict({
    'microbiology': latest_dir / 'icare_microbiology_anon.csv',
    'pharmacy': latest_dir / 'icare_pharmacy_prescribing_anon.csv',
    'problems': latest_dir / 'icare_problems_anon.csv',
    'diagnoses': latest_dir / 'icare_episodes_diagnosis_anon.csv'
})

📂 Changed working directory to project root: C:\Users\kelda\Desktop\repositories\github\dm-esbl
⚙️ Loaded default config from: C:\Users\kelda\Desktop\repositories\github\dm-esbl\src\icare_risk\config\data_config.yaml
⚙️ No override config provided. Running solely on defaults.
⚙️ Loaded default config from: C:\Users\kelda\Desktop\repositories\github\dm-esbl\src\icare_risk\config\feature_config.yaml
⚠️ Warning: Override file '../config/local_feature_config.yaml' not found. Using defaults.
🔄 Active Data Source: [SYNTHETIC] -> Reading from '2026-07-30_151244'
Using data from: 2026-07-30_151244
  -> Vitals pivoted. Shape: (12000, 10)
  -> Labs pivoted. Shape: (6665, 8)
  -> Combined Time-Series Shape: (12000, 16)


In [8]:
def test_random_feature(df, **kwargs):
    """Generates a random binary flag (0 or 1) for each row."""
    np.random.seed(kwargs.get('seed', 42))
    return np.random.randint(0, 2, size=len(df))

# Run it in your notebook
df_ts['random_test_flag'] = test_random_feature(df_ts, seed=123)
display(df_ts[['patient_id', 'date', 'random_test_flag']].head(5))

,patient_id,date,random_test_flag
0,10001,2021-09-20 00:00:00,0
1,10001,2021-09-20 04:00:00,1
2,10001,2021-09-20 08:00:00,0
3,10001,2021-09-20 12:00:00,0
4,10001,2021-09-20 16:00:00,0


In [15]:
def test_threshold_feature(df, **kwargs):
    """Flags rows where a vital sign crosses a specific threshold."""
    flag = pd.Series(0, index=df.index)
    col = kwargs.get('target_col', 'hr')
    threshold = kwargs.get('threshold', 90.0)

    if col in df.columns:
        val = pd.to_numeric(df[col], errors='coerce')
        flag.loc[val > threshold] = 1

    return flag.values

# Run it in your notebook
df_ts['tachycardia_flag'] = test_threshold_feature(
    df_ts, target_col='hr', threshold=90.0
)
display(df_ts[['patient_id', 'date', 'hr', 'tachycardia_flag']].head(5))

,patient_id,date,hr,tachycardia_flag
0,10001,2021-09-20 00:00:00,31.23,0
1,10001,2021-09-20 04:00:00,NaN,0
2,10001,2021-09-20 08:00:00,120.72,1
3,10001,2021-09-20 12:00:00,191.98,1
4,10001,2021-09-20 16:00:00,37.04,0


In [12]:
from icare_risk.phenotypes import _patient_has_historical_codes

def test_relational_condition_feature(df, **kwargs):
    """Checks external relational context tables (like problems or diagnoses) for target codes."""
    context = kwargs.get('context_dfs', {})
    target_codes = kwargs.get('target_codes', ['I10']) # e.g., Hypertension

    # Leverages the robust helper from phenotypes.py
    has_condition = _patient_has_historical_codes(
        df=df,
        context_df=context.get('problems'), # Triggers Lazy Load for problems table
        patient_col='SUBJECT',
        code_col='PROBLEM_CODE',
        target_codes=target_codes
    )

    return has_condition.astype(int).values

# Run it in your notebook using your loaded 'contexts' dictionary
df_ts['hypertension_hx_flag'] = test_relational_condition_feature(
    df_ts,
    context_dfs=contexts,
    target_codes=['I10', 'I11']
)

# Inspect results for patients who actually matched the historical codes
display(df_ts[df_ts['hypertension_hx_flag'] == 1][['patient_id', 'date', 'hypertension_hx_flag']].head(10))

,patient_id,date,hypertension_hx_flag
60,10002,2020-11-04 00:00:00,1
61,10002,2020-11-04 04:00:00,1
62,10002,2020-11-04 08:00:00,1
63,10002,2020-11-04 12:00:00,1
64,10002,2020-11-04 16:00:00,1
65,10002,2020-11-04 20:00:00,1
66,10002,2020-11-05 00:00:00,1
67,10002,2020-11-05 04:00:00,1
68,10002,2020-11-05 08:00:00,1
69,10002,2020-11-05 12:00:00,1
